# Load Data

In [1]:
import numpy as np

loaded = np.load("ecg_dataset.npz")
X = loaded["x"]
y = loaded["y"]

In [2]:
X_2d = X.transpose(0, 2, 1)  # shape is (n_samples, 12, 5000)

# Step 2: Add a channel dimension (like grayscale image)
X_2d = X_2d[..., np.newaxis]

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_2d, y, test_size=0.2, random_state=42, stratify=y)

# Model Creation

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_2d_cnn(input_shape=(12, 5000, 1)):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
    ])
    return model

In [7]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense

cnn_model = build_2d_cnn() 
# Clone base
feature_extractor = cnn_model

# Add output layer (e.g., binary classification for heart failure)
output = Dense(1, activation='sigmoid')(feature_extractor.output)
training_model = Model(inputs=feature_extractor.input, outputs=output)

training_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
        monitor='val_loss',       # Track validation loss
        patience=3,               # Stop after 3 epochs with no improvement
        restore_best_weights=True
    )
    

In [ ]:
training_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop])

Epoch 1/100
340/340 [==============================] - 831s 2s/step - loss: 0.7283 - accuracy: 0.7338 - val_loss: 0.4445 - val_accuracy: 0.8213
Epoch 2/100
340/340 [==============================] - 538s 2s/step - loss: 0.3987 - accuracy: 0.8317 - val_loss: 0.4220 - val_accuracy: 0.8271
Epoch 3/100
340/340 [==============================] - 420s 1s/step - loss: 0.3028 - accuracy: 0.8717 - val_loss: 0.4365 - val_accuracy: 0.8147
Epoch 4/100
340/340 [==============================] - 420s 1s/step - loss: 0.1984 - accuracy: 0.9223 - val_loss: 0.5040 - val_accuracy: 0.8197
Epoch 5/100
340/340 [==============================] - 417s 1s/step - loss: 0.1086 - accuracy: 0.9608 - val_loss: 0.6389 - val_accuracy: 0.8056
